In [1]:
from stipy import *

In [2]:
server = connect(DeviceID("STI Server", "localhost", 0), "192.168.1.109:2809")

++++ add( localhost/0/STI Server )


In [2]:
server = connect(DeviceID("TestDevice", "localhost", 0), "192.168.1.109:2809")

++++ add( localhost/0/TestDevice )


In [3]:
server.getChannelManager().getChannel(3).metadata()

{'vectorFormat': (Number,String,Boolean),
 'color': green,
 'valueHint': [Frequency (MHz), name, enable]}

---- remove( localhost/0/TestDevice )


In [51]:
server = connect(DeviceID("Digital Out", "sr-timing2", 2), "192.168.1.247:2809")

In [52]:
server.getDeviceCollection().getIDs()

[HOGAN-Book3/0/STIPy:HOGAN-Book3:2025-12-17:21_02_56_369868400,
 sr-timing2/8/FPGA Trigger,
 sr-timing2/9/FPGA Memory]

In [53]:
def test_fpga():
    digital = dev("Digital Out", "sr-timing2", 2)
    ns = 1
    us = 1000
    ms = 1000000
    s = 1000000000
    
    time = 10*us

    event(ch(digital, 0), time, True)
    event(ch(digital, 0), time + 20*us, False)
    event(ch(digital, 0), time + 30*us, True)
    event(ch(digital, 0), time + 32*us, False)
    event(ch(digital, 0), time + 36*us, True)

def test_fpga2():
    digital = dev("Digital Out", "sr-timing2", 2)
    ns = 1
    us = 1000
    ms = 1000000
    s = 1000000000
    
    time = 11*us

    event(ch(digital, 0), time, True)
    event(ch(digital, 0), time + 20*us, False)
    event(ch(digital, 0), time + 40*us, True)
    # event(ch(digital, 0), time + 41*us, True)
    # event(ch(digital, 0), time + 42*us, True)


In [54]:
shot = server.makeshot(test_fpga)

In [76]:
shot2 = server.makeshot(test_fpga2)

In [48]:
shot.rootgroup().events()

[event(Time=10us|0ns, Channel=dev(sr-timing2/2/Digital Out).ch(0), Value=1),
 event(Time=30us|0ns, Channel=dev(sr-timing2/2/Digital Out).ch(0), Value=0),
 event(Time=40us|0ns, Channel=dev(sr-timing2/2/Digital Out).ch(0), Value=1),
 event(Time=42us|0ns, Channel=dev(sr-timing2/2/Digital Out).ch(0), Value=0),
 event(Time=46us|0ns, Channel=dev(sr-timing2/2/Digital Out).ch(0), Value=1)]

In [55]:
tick = server.parse(shot)

In [32]:
tick.messages()

[]

In [56]:
playtick = server.play(tick)

In [58]:
playtick

<ResultTicket | Running | sid:Jason@HOGAN-Book3#21_03_15_623873500>

In [59]:
server.cancelAll()

In [88]:
tick1 = server.parse(shot)
tick1.messages()

[]

In [93]:
playtick = server.play(tick1)

In [27]:
playtick

<ResultTicket | Canceled | sid:Jason@HOGAN-Book3#20_53_43_125942600>

In [84]:
tick2 = server.parse(shot2)
tick2.messages()

[]

In [92]:
playtick = server.play(tick2)

In [20]:
playtick.status()

<TicketStatus.Complete: 1>

In [9]:
import os
import platform
import socket
import ipaddress
import subprocess

def _default_route_ip():
    s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    s.connect(("8.8.8.8", 1))
    return s.getsockname()[0]

def _default_iface():
    system = platform.system().lower()
    if system == "linux":
        try:
            with open("/proc/net/route", "r", encoding="ascii") as f:
                for line in f.readlines()[1:]:
                    fields = line.strip().split()
                    if len(fields) >= 2 and fields[1] == "00000000":
                        return fields[0]
        except OSError:
            return None
    if system == "darwin":
        try:
            import netifaces  # optional
            return netifaces.gateways()["default"][netifaces.AF_INET][1]
        except Exception:
            pass
        try:
            out = subprocess.check_output(["route", "-n", "get", "default"], text=True)
            for line in out.splitlines():
                line = line.strip()
                if line.startswith("interface:"):
                    return line.split(":", 1)[1].strip()
        except Exception:
            return None
    return None

def get_local_ip_address():
    env_ip = os.getenv("STI3_PUBLISH_IP")
    if env_ip:
        return env_ip

    system = platform.system().lower()
    if system == "windows":
        return _default_route_ip()

    try:
        import fcntl
        import struct
    except ModuleNotFoundError:
        return _default_route_ip()

    def _ifaddr(ifname):
        s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
        try:
            return socket.inet_ntoa(
                fcntl.ioctl(
                    s.fileno(),
                    0x8915,  # SIOCGIFADDR
                    struct.pack("256s", ifname[:15].encode("utf-8"))
                )[20:24]
            )
        except OSError:
            return None

    env_subnet = os.getenv("STI3_PUBLISH_SUBNET")
    if env_subnet:
        net = ipaddress.ip_network(env_subnet, strict=False)
        for _, ifname in socket.if_nameindex():
            ip = _ifaddr(ifname)
            if ip and ipaddress.ip_address(ip) in net:
                return ip

    default_iface = _default_iface()
    print("Default interface:", default_iface)
    for _, ifname in socket.if_nameindex():
        if ifname in ("lo", "lo0"):
            continue
        ip = _ifaddr(ifname)
        if ip and ifname == default_iface:
            return ip

    return _default_route_ip()


In [10]:
get_local_ip_address()

Default interface: eno2


'192.168.1.109'

In [4]:
_default_route_ip()

'192.168.1.109'

In [8]:
_default_iface()

'eno2'

In [1]:
import os
import platform
import socket

def _default_route_ip():
    s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    s.connect(("8.8.8.8", 1))
    return s.getsockname()[0]

def _in_container():
    if os.path.exists("/.dockerenv"):
        return True
    try:
        with open("/proc/1/cgroup", "r", encoding="ascii") as f:
            data = f.read()
        return "docker" in data or "containerd" in data
    except OSError:
        return False

def publish_ip():
    env_ip = os.getenv("STI3_PUBLISH_IP")
    if env_ip:
        return env_ip

    if platform.system().lower() == "windows":
        return _default_route_ip()

    try:
        import fcntl, struct, ipaddress
    except ModuleNotFoundError:
        return _default_route_ip()

    def _ifaddr(ifname):
        s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
        try:
            return socket.inet_ntoa(
                fcntl.ioctl(
                    s.fileno(),
                    0x8915,  # SIOCGIFADDR
                    struct.pack("256s", ifname[:15].encode("utf-8"))
                )[20:24]
            )
        except OSError:
            return None

    def _default_iface_linux():
        try:
            with open("/proc/net/route", "r", encoding="ascii") as f:
                for line in f.readlines()[1:]:
                    fields = line.strip().split()
                    if len(fields) >= 2 and fields[1] == "00000000":
                        return fields[0]
        except OSError:
            return None

    # Optional subnet override (Linux)
    env_subnet = os.getenv("STI3_PUBLISH_SUBNET")
    if env_subnet:
        net = ipaddress.ip_network(env_subnet, strict=False)
        for _, ifname in socket.if_nameindex():
            ip = _ifaddr(ifname)
            if ip and ipaddress.ip_address(ip) in net:
                return ip

    ips = {}
    for _, ifname in socket.if_nameindex():
        if ifname == "lo":
            continue
        ip = _ifaddr(ifname)
        if ip:
            ips[ifname] = ip

    if not ips:
        return _default_route_ip()

    if len(ips) == 1:
        return next(iter(ips.values()))

    default_iface = _default_iface_linux()
    if _in_container() and default_iface in ips:
        for ifname, ip in ips.items():
            if ifname != default_iface:
                return ip

    if default_iface in ips:
        return ips[default_iface]

    return next(iter(ips.values()))


In [4]:
publish_ip()

'192.168.1.109'